# Indeed Job Scraping Notebook

Notebook workflow for collecting Indeed postings with checkpoint support.


## Step 1: System Check

In [1]:
import sys
import platform

print("SYSTEM CHECK")
print("="*70)
print(f"OS: {platform.system()}")
print(f"Python: {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")

if sys.version_info < (3, 7):
    print("\n❌ ERROR: Python 3.7+ required")
else:
    print("\n✅ Python version OK")


SYSTEM CHECK
OS: Windows
Python: 3.11.13

✅ Python version OK


## Step 2: Install Dependencies


In [2]:
print("Installing required libraries...\n")
!pip install --quiet pandas beautifulsoup4 camoufox camoufox-captcha
print("\n✅ Installation complete!")

Installing required libraries...


✅ Installation complete!


## Step 2.1: Verify Dependencies

In [3]:
import importlib

print("DEPENDENCY CHECK")
print("="*70)

required = ['pandas', 'bs4', 'camoufox', 'camoufox_captcha']
all_good = True

for lib in required:
    try:
        importlib.import_module(lib)
        print(f"✅ {lib}")
    except ImportError:
        print(f"❌ {lib} - MISSING")
        all_good = False

print("="*70)
if all_good:
    print("✅ All dependencies installed!")
else:
    print("❌ Some dependencies missing - re-run Step 2")

DEPENDENCY CHECK
✅ pandas
✅ bs4
✅ camoufox
✅ camoufox_captcha
✅ All dependencies installed!


## Step 3: Configure Search

**Customize these settings:**

- `POSITION`: Job title (e.g., "data analyst", "python developer")
- `LOCATION`: Location (e.g., "remote", "New York, NY")
- `MAX_JOBS`: How many jobs to scrape (recommended: 100-1000)

**Time estimates:**
- 100 jobs ≈ 25 minutes
- 500 jobs ≈ 2 hours
- 1000 jobs ≈ 3-4 hours

In [ ]:
# ===== CONFIGURE YOUR SEARCH =====

POSITION = "python analyst"
LOCATION = "remote"
MAX_JOBS = 10

# =================================

print("="*70)
print("SEARCH CONFIGURATION")
print("="*70)
print(f"Position: {POSITION}")
print(f"Location: {LOCATION}")
print(f"Max Jobs: {MAX_JOBS}")
print(f"\nEstimated time: {MAX_JOBS * 0.25:.0f}-{MAX_JOBS * 0.35:.0f} minutes")
print("="*70)
print("\nTIP: Start with 100 jobs, then scale up")
print("TIP: This cell can run in background - minimize and continue working")
print("="*70)

## Step 4: Run Scraper

### What happens:
1. **Checkpoint detection** - Automatically resumes if you've run this search before
2. **Browser opens** - Don't close it!
3. **Progress updates** - Every 50 jobs
4. **Auto-save** - Progress saved every 50 jobs
5. **Final CSV** - Saved when complete

### Important:
-  **Can be interrupted** - Just re-run to resume
- **Auto-saved** - Progress is never lost
- **Multiple searches** - Each search has separate checkpoints
-  **Don't close browser** - It's working!

### If something goes wrong:
- Just re-run this cell - it will resume automatically
- Check error messages for guidance


In [7]:
import subprocess
import sys
import os
from datetime import datetime

# Locate script
SCRIPT_NAME = "indeed_scraper_camoufox.py"
SCRIPT_PATH = os.path.join(os.getcwd(), SCRIPT_NAME)

if not os.path.exists(SCRIPT_PATH):
    print("="*70)
    print("❌ ERROR: Script not found!")
    print("="*70)
    print(f"\nLooking for: {SCRIPT_NAME}")
    print(f"In directory: {os.getcwd()}")
    print("\n📥 Download the script and place it in the same folder as this notebook.")
    print("\nFiles in current directory:")
    for f in os.listdir('.'):
        if f.endswith('.py'):
            print(f"  - {f}")
else:
    print("="*70)
    print("STARTING SCRAPER")
    print("="*70)
    print(f"Search: {POSITION} in {LOCATION}")
    print(f"Target: {MAX_JOBS} jobs")
    print(f"Start: {datetime.now().strftime('%H:%M:%S')}")
    print("\n" + "="*70)
    print("SCRAPER OUTPUT")
    print("="*70 + "\n")
    
    try:
        result = subprocess.run(
            [sys.executable, SCRIPT_PATH, POSITION, LOCATION, str(MAX_JOBS)],
            capture_output=True,
            text=True
        )
        
        print(result.stdout)
        
        if result.stderr:
            # Filter out harmless warnings
            stderr_lines = result.stderr.split('\n')
            errors = [line for line in stderr_lines if line and 'charmap' not in line and 'Downloading GeoIP' not in line]
            if errors:
                print("\n" + "="*70)
                print("WARNINGS/ERRORS")
                print("="*70)
                for error in errors:
                    print(error)
        
        print("\n" + "="*70)
        print(f"End: {datetime.now().strftime('%H:%M:%S')}")
        print("="*70)
        
        if result.returncode == 0:
            print("\n✅ SCRAPING COMPLETED!")
        else:
            print(f"\n⚠️ Scraper exited with code: {result.returncode}")
            print("\n🔄 Try running this cell again - it will resume from checkpoint")
    
    except Exception as e:
        print("\n" + "="*70)
        print("❌ ERROR")
        print("="*70)
        print(f"\n{e}")


STARTING SCRAPER


NameError: name 'POSITION' is not defined

## Step 5: Load Results

In [10]:
import pandas as pd
import glob
import os

print("="*70)
print("LOADING RESULTS")
print("="*70)

# Find most recent CSV (not checkpoint)
csv_files = [f for f in glob.glob("*.csv") if not f.startswith('checkpoint')]

if csv_files:
    latest_file = max(csv_files, key=os.path.getctime)
    print(f"\n✅ Found: {latest_file}")
    
    df = pd.read_csv(latest_file)
    
    print(f"\n{'='*70}")
    print("DATA SUMMARY")
    print("="*70)
    print(f"Total jobs: {len(df)}")
    
    jobs_with_salary = (df['Salary'] != 'N/A').sum()
    print(f"Jobs with salary: {jobs_with_salary} ({(jobs_with_salary/len(df)*100):.1f}%)")
    print(f"Unique companies: {df['Company'].nunique()}")
    print(f"Unique locations: {df['Location'].nunique()}")
    
    print(f"\n{'='*70}")
    print("PREVIEW (First 5 rows)")
    print("="*70)
    display(df.head())
    
    print("\n✅ Data loaded successfully!")
    
else:
    print("\n❌ No results found!")
    print("\nPossible reasons:")
    print("  1. Scraper hasn't completed (check Step 4)")
    print("  2. File was deleted or moved")
    print("  3. Scraper failed to save output")
    df = None

LOADING RESULTS

✅ Found: 2025-12-04_18-39_business analyst_remote.csv

DATA SUMMARY
Total jobs: 16
Jobs with salary: 16 (100.0%)
Unique companies: 14
Unique locations: 10

PREVIEW (First 5 rows)


,Title,Company,Location,Rating,Date,Salary,Description,Links
0,Business Functional Analyst,IDR Inc.,"Remote in Olympia, WA 98507",NaN,NaN,$50an hour,IDR is seeking a\nBusiness and Functional Anal...,https://www.indeed.com/pagead/clk?mo=r&ad=-6NY...
1,Senior Business Analyst (Remote),Biolife Plasma Services,"Remote in Bannockburn, IL 60015",NaN,NaN,"$86,500 - $135,960a year","By clicking the “Apply” button, I understand t...",https://www.indeed.com/pagead/clk?mo=r&ad=-6NY...
2,Strategy & Planning Analyst,Enlyte,Remote,NaN,NaN,"$80,000 - $105,000a year","Company Overview:\nAt Enlyte, we combine innov...",https://www.indeed.com/pagead/clk?mo=r&ad=-6NY...
3,IT Business Intelligence Analyst,Implus,Hybrid work,NaN,NaN,NaN,"Implus Footcare, LLC\nis an industry-leading g...",https://www.indeed.com/rc/clk?jk=4f6552f0ba748...
4,IT Business Intelligence Analyst,Implus,Hybrid work,NaN,NaN,NaN,Description not found,https://www.indeed.com/viewjob?jk=123456789abc...



✅ Data loaded successfully!


## Step 6: Detailed Analysis

In [11]:
if df is not None:
    print("="*70)
    print("DETAILED ANALYSIS")
    print("="*70)
    
    # Top Companies
    print("\n📊 TOP 10 COMPANIES")
    print("-"*70)
    top_companies = df['Company'].value_counts().head(10)
    for i, (company, count) in enumerate(top_companies.items(), 1):
        print(f"{i:2}. {company:40} ({count} jobs)")
    
    # Top Locations
    print("\n📍 TOP 10 LOCATIONS")
    print("-"*70)
    top_locations = df['Location'].value_counts().head(10)
    for i, (location, count) in enumerate(top_locations.items(), 1):
        print(f"{i:2}. {location:40} ({count} jobs)")
    
    # Salary Info
    jobs_with_salary = df[df['Salary'] != 'N/A']
    
    if len(jobs_with_salary) > 0:
        print("\n💰 SALARY INFORMATION")
        print("-"*70)
        print(f"Jobs with salary: {len(jobs_with_salary)}")
        print(f"\nSample salaries:")
        for i, salary in enumerate(jobs_with_salary['Salary'].head(10), 1):
            print(f"{i:2}. {salary}")
    else:
        print("\n💰 No salary information in this dataset")
        print("   (This is normal - many employers don't post salaries)")
else:
    print("❌ No data to analyze - run Step 5 first")

DETAILED ANALYSIS

📊 TOP 10 COMPANIES
----------------------------------------------------------------------
 1. Implus                                   (2 jobs)
 2. Contact Government Services, LLC         (2 jobs)
 3. Biolife Plasma Services                  (1 jobs)
 4. Enlyte                                   (1 jobs)
 5. OM Group Inc.                            (1 jobs)
 6. IDR Inc.                                 (1 jobs)
 7. Lincare Inc.                             (1 jobs)
 8. MCC                                      (1 jobs)
 9. Pyramid Consulting, Inc                  (1 jobs)
10. Kaztronix                                (1 jobs)

📍 TOP 10 LOCATIONS
----------------------------------------------------------------------
 1. Hybrid work                              (6 jobs)
 2. Remote                                   (2 jobs)
 3. Remote in Bannockburn, IL 60015          (1 jobs)
 4. Remote in Olympia, WA 98507              (1 jobs)
 5. Remote in Clearwater, FL 33764          

## Step 7: View Jobs with Salaries

In [ ]:
if df is not None:
    jobs_with_salary = df[df['Salary'] != 'N/A'][['Title', 'Company', 'Location', 'Salary']]
    
    if len(jobs_with_salary) > 0:
        print("="*70)
        print(f"JOBS WITH SALARY INFO ({len(jobs_with_salary)} jobs)")
        print("="*70)
        display(jobs_with_salary)
    else:
        print("No jobs with salary information found")
else:
    print("❌ No data loaded")


## Advanced: Run Multiple Searches



In [8]:
import subprocess
import sys
import os
from datetime import datetime

# Locate script
SCRIPT_NAME = "indeed_scraper_camoufox.py"
SCRIPT_PATH = os.path.join(os.getcwd(), SCRIPT_NAME)

In [12]:
# Define multiple searches
searches = [
    ## DATA ANALYST SEARCHES
    #{"position": "data analyst", "location": "San Francisco, CA", "max_jobs": 100},
    #{"position": "data analyst", "location": "New York, NY", "max_jobs": 100},
    #{"position": "data analyst", "location": "Seattle, WA", "max_jobs": 100},
    #{"position": "data analyst", "location": "remote", "max_jobs": 200},
    ## DATA SCIENTIST SEARCHES
    #{"position": "data scientist", "location": "San Francisco, CA", "max_jobs": 100},
    {"position": "data scientist", "location": "New York, NY", "max_jobs": 100},
    #{"position": "data scientist", "location": "Seattle, WA", "max_jobs": 100},
    {"position": "data scientist", "location": "remote", "max_jobs": 200},
    ## MACHINE LEARNING ENGINEER SEARCHES
    #{"position": "machine learning engineer", "location": "San Francisco, CA", "max_jobs": 100},
    #{"position": "machine learning engineer", "location": "New York, NY", "max_jobs": 100},
    {"position": "machine learning engineer", "location": "Seattle, WA", "max_jobs": 100},
    #{"position": "machine learning engineer", "location": "remote", "max_jobs": 200},
    ## BUSINESS ANALYST SEARCHES
    #{"position": "business analyst", "location": "San Francisco, CA", "max_jobs": 100},
    #{"position": "business analyst", "location": "New York, NY", "max_jobs": 100},
    #{"position": "business analyst", "location": "Seattle, WA", "max_jobs": 100},
    #{"position": "business analyst", "location": "remote", "max_jobs": 200},
]

# Run each search
for i, search in enumerate(searches, 1):
    print(f"\n{'='*70}")
    print(f"SEARCH {i}/{len(searches)}")
    print("="*70)
    print(f"Position: {search['position']}")
    print(f"Location: {search['location']}")
    print(f"Max Jobs: {search['max_jobs']}")
    print("="*70)
    
    # Note: Each search will have its own checkpoint files
    # You can interrupt and resume each search independently
    
    result = subprocess.run(
        [sys.executable, SCRIPT_PATH, search['position'], search['location'], str(search['max_jobs'])],
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print(f"\n✅ Search {i} completed!")
    else:
        print(f"\n⚠️ Search {i} had issues - check output above")

print("\n" + "="*70)
print("ALL SEARCHES COMPLETE!")
print("="*70)


SEARCH 1/3
Position: data scientist
Location: New York, NY
Max Jobs: 100

✅ Search 1 completed!

SEARCH 2/3
Position: data scientist
Location: remote
Max Jobs: 200

✅ Search 2 completed!

SEARCH 3/3
Position: machine learning engineer
Location: Seattle, WA
Max Jobs: 100

✅ Search 3 completed!

ALL SEARCHES COMPLETE!
